## Future Pricing

In [147]:
import pandas as pd

df = pd.read_excel("data/combined.xlsx")
df.columns = df.columns.str.strip()

# data = df.iloc[0:20]
data = df.iloc[21:].copy()
data = data.iloc[::-1].reset_index(drop=True)
data.head()

,Symbol,Date,Expiry,Open,High,Low,Close,LTP,Settle Price,No. of contracts,Turnover * IN‚ INR Lakhs,Open Int,Change in OI,Underlying Value
0,HDFCLIFE,2026-01-01,2026-03-30,763.30,763.85,762.2,762.20,762.20,762.20,13.0,109.15,22000.0,13200,750.10
1,HDFCLIFE,2026-01-02,2026-03-30,764.00,769.95,764.0,767.05,767.05,764.80,6.0,50.63,24200.0,2200,754.85
2,HDFCLIFE,2026-01-05,2026-03-30,775.05,777.20,772.4,772.40,772.40,772.40,10.0,85.19,26400.0,2200,759.30
3,HDFCLIFE,2026-01-06,2026-03-30,783.75,788.15,782.1,784.50,784.50,787.65,11.0,95.06,31900.0,5500,777.85
4,HDFCLIFE,2026-01-07,2026-03-30,786.00,786.00,783.7,784.35,784.40,784.35,4.0,34.52,31900.0,-,772.35


In [148]:
lot_size = 1100
buy_price = data["Settle Price"][0]
print(buy_price)
invest_avail = 4500000

one_lot_price_jan1 = lot_size * buy_price
future_contr = int(invest_avail / (one_lot_price_jan1))

invest = future_contr * one_lot_price_jan1
init_marg = 0.3 * invest
maint_marg = 0.2 * invest

buff = 5000000 - (invest+init_marg)

risk_free_91 = 5.47/100
risk_free_182 = 5.66/100


if buff >= 0:
    buff = buff
    borrows = 0
else:
    borrows = buff
    buff = 0


762.2


In [149]:
print(future_contr)
print(one_lot_price_jan1)
print(invest)
print(init_marg)
print(maint_marg)
print(borrows)

5
838420.0
4192100.0
1257630.0
838420.0
-449730.0


In [150]:
def interest(risk_free_, borrow = 0, profit = 0, mclr_intr = (9.85/100)): ##which risk free interest to use??
    if(borrow == 0 and profit <= 0):
        return 0
    elif(profit > 0 and borrow == 0):
        return profit*((1 + risk_free_)**(1/365)-1)
    elif(borrow < 0 and profit <= 0):
        return borrow*((1 + mclr_intr)**(1/365)-1)
    elif(borrow < 0 and profit > 0):
        return profit*((1 + risk_free_)**(1/365)-1) + borrow*((1 + mclr_intr)**(1/365)-1) 

In [151]:
rows = []
interest_due = 0
profits = 0

margin_amt = init_marg
data["Prev_Settle_Price"] = data["Settle Price"].shift(1)

for row, colmn in data.iterrows():
    margin_call = 0
    interest_profit = 0
    day = colmn["Date"]
    settelment = colmn["Settle Price"]
    prev_day_price = colmn["Prev_Settle_Price"]

    if pd.isna(prev_day_price):
        price_diff = 0
    else:
        price_diff = settelment - prev_day_price
    
    daily_profits = price_diff * one_lot_price_jan1

    margin_amt = margin_amt + daily_profits
    
    if(margin_amt < maint_marg):
        margin_call = init_marg - margin_amt
        borrows = borrows - margin_call
        margin_amt = init_marg
    
    if(margin_amt > init_marg):
        interest_profit = margin_amt - init_marg

    rows.append([
        day,
        settelment,
        prev_day_price,
        price_diff,
        daily_profits,
        margin_amt,
        borrows,
        interest_profit,
        interest_due,
        margin_call,
    ])

    interest_due = interest(risk_free_= risk_free_91, profit= interest_profit, borrow= borrows)

    log_sheet = pd.DataFrame(rows, columns=[
    "Day",
    "Settelment",
    "Prev_Settelment",
    "Daily_Price_Change",
    "Daily_Profits",
    "Margin_Acct",
    "Borrows",
    "Interest_Profit",
    "Interest_Flow",
    "Margin_call"
])

In [152]:
log_sheet.to_excel("logs/logs_feb.xlsx")
log_sheet.head()

,Day,Settelment,Prev_Settelment,Daily_Price_Change,Daily_Profits,Margin_Acct,Borrows,Interest_Profit,Interest_Flow,Margin_call
0,2026-01-01,762.20,NaN,0.00,0.0,1257630.0,-449730.0,0.0,0.000000,0.0
1,2026-01-02,764.80,762.20,2.60,2179892.0,3437522.0,-449730.0,2179892.0,-115.768762,0.0
2,2026-01-05,772.40,764.80,7.60,6371992.0,9809514.0,-449730.0,8551884.0,202.317804,0.0
3,2026-01-06,787.65,772.40,15.25,12785905.0,22595419.0,-449730.0,21337789.0,1132.109306,0.0
4,2026-01-07,784.35,787.65,-3.30,-2766786.0,19828633.0,-449730.0,18571003.0,2997.809358,0.0


In [153]:
sum_intr_flow = log_sheet["Interest_Flow"].sum()
total_gain = log_sheet["Margin_Acct"].iloc[-1] + log_sheet["Borrows"].iloc[-1] + sum_intr_flow
total_gain

np.float64(-19996614.21228883)

## Forward Pricing

In [154]:
import pandas as pd

df = pd.read_excel("data/combined.xlsx")
df.columns = df.columns.str.strip()

data = df.iloc[0:20]
# data = df.iloc[21:].copy()
data = data.iloc[::-1].reset_index(drop=True)
data.head()

,Symbol,Date,Expiry,Open,High,Low,Close,LTP,Settle Price,No. of contracts,Turnover * IN‚ INR Lakhs,Open Int,Change in OI,Underlying Value
0,HDFCLIFE,2026-01-01,2026-02-24,759.35,760.60,756.00,758.10,757.40,758.10,24.0,200.14,359700.0,4400,750.10
1,HDFCLIFE,2026-01-02,2026-02-24,759.00,765.20,759.00,761.95,761.75,761.95,43.0,360.76,369600.0,9900,754.85
2,HDFCLIFE,2026-01-05,2026-02-24,766.35,773.05,765.95,767.15,767.50,767.15,112.0,948.15,378400.0,8800,759.30
3,HDFCLIFE,2026-01-06,2026-02-24,779.20,788.60,776.35,783.70,783.95,783.70,188.0,1617.48,388300.0,9900,777.85
4,HDFCLIFE,2026-01-07,2026-02-24,783.70,784.30,778.60,780.00,780.40,780.00,68.0,584.22,396000.0,7700,772.35


In [155]:
lot_size = 1100
buy_price = data["Settle Price"][0]
invest_avail = 4500000

one_lot_price_jan1 = lot_size * buy_price
future_contr = int(invest_avail / (one_lot_price_jan1))

invest = future_contr * one_lot_price_jan1

risk_free_91 = 5.47/100

In [156]:
import math

time = (data["Date"].iloc[-1] - data["Date"][0]).days
time = time/365
S_o = data["Underlying Value"][0]
F_o = S_o * math.exp(risk_free_91 * time)
forw_profit = F_o - data["Settle Price"].iloc[-1]
total_forw_profit = forw_profit * lot_size * future_contr
total_forw_profit

np.float64(109268.77143246894)

In [157]:
data["Settle Price"].iloc[-1]

np.float64(733.5)